In [33]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import ray
from ray import tune
from ray.tune.schedulers.pb2 import PB2, PopulationBasedTraining
from ray.tune import Checkpoint, run, sample_from 
from tensorboardX import SummaryWriter

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SOURCES_ROOT = os.path.join(PROJECT_ROOT, 'Sources')
PYTHONPATH_VALUE = os.pathsep.join([PROJECT_ROOT, SOURCES_ROOT])

RAY_EXCLUDES = [
    '.git/',
    '.venv/',
    'Dataset/',
    'Results/',
    'SampleVideos/',
    'Sources/Tests/deepmimo_scenarios/',
    'Sources/Tests/deepmimo_scenarios/*.zip',
]

for path in [PROJECT_ROOT, SOURCES_ROOT]:
    if path not in sys.path:
        sys.path.insert(0, path)

if PYTHONPATH_VALUE not in os.environ.get('PYTHONPATH', ''):
    existing = os.environ.get('PYTHONPATH', '')
    os.environ['PYTHONPATH'] = f"{PYTHONPATH_VALUE}{os.pathsep}{existing}" if existing else PYTHONPATH_VALUE

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from 'c:\\Users\\es25591\\Workspace\\CacheVideoPredict360\\Sources\\Common\\utils.py'>

In [34]:
def resolve_torch_device() -> torch.device:
    if torch.cuda.is_available():
        try:
            _ = torch.empty(1, device="cuda")
            return torch.device("cuda")
        except Exception:
            pass
    return torch.device("cpu")


device = resolve_torch_device()

UserTransition = datatypes.UserTransition
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey

cfg = config.Config()
cfg.filename = f"drl_kulkani_softup_lrdecay{cfg.learning_rate_decay}_c{cfg.cache_size}_ar{cfg.arrival_rate}_z{cfg.zipf_alpha}.csv"

debugger = debugger.debug

In [ ]:
class DrlPolicy(CachePolicy):
    def __init__(self, cfg: Any = None):
        self.cfg = cfg
        self.cur_size = 0

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def get(self, key: CacheKey) -> Any:
        return self.cache.get(key, None)

    def put(self, key: int, value: Any, size: int) -> list:
        """
        key   -> slot index
        value -> (video_id, tiles)
        size  -> fixed as 1 slot
        """
        evicted = []
        slot = key
        new_video, _ = value

        if new_video in self.video_idx:
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return evicted

        self.video_idx[slot] = new_video
        self.tile_idx[slot] = [-1] * self.cfg.viewport

        self.cur_size = sum(1 for v in self.video_idx if v != -1)
        return evicted

    def contains(self, key: CacheKey) -> bool:
        return key in self.video_idx

    def remove(self, key: CacheKey) -> bool:
        if key in self.video_idx:
            idx = self.video_idx.index(key)
            self.video_idx[idx] = -1
            self.tile_idx[idx] = [-1] * self.cfg.viewport
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return True
        return False

    def clear(self) -> None:
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]
        self.cur_size = 0

    def keys(self):
        return self.video_idx
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def update_size(self):
        self.cur_size = sum(1 for v in self.video_idx if v != -1)

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [36]:
class BaseWorker:
    def __init__(self, cfg):
        self.cfg = cfg
        self.step = 0
        self.n_step = cfg.n_step
        self.state_dim = cfg.state_dim_meta
        self.num_actions = cfg.action_dim_meta

        self.gamma = cfg.gamma
        self.epsilon = cfg.epsilon_start
        self.epsilon_min = cfg.epsilon_min
        self.epsilon_decay = cfg.epsilon_decay
        self.batch_size = cfg.batch_size
        self.tau = cfg.tau
        self.device = resolve_torch_device()

        self.buffer = NStepReplayBuffer(cfg.buffer_capacity, self.n_step, self.gamma)

        self.policy_net = QNetwork(self.state_dim, self.num_actions).to(self.device)
        self.target_net = QNetwork(self.state_dim, self.num_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg.learning_rate)

        self.scheduler = optim.lr_scheduler.ExponentialLR(
            self.optimizer, gamma=cfg.learning_rate_decay
        )

        self.loss_fn = nn.MSELoss()
        self.nb_interval = cfg.nb_interval

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.num_actions - 1)

        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32).to(self.device).unsqueeze(0)
            qvals = self.policy_net(state_t)
            return qvals.argmax().item()

    def remember(self, s, a, r, ns, done):
        self.buffer.push(s, a, r, ns, done)

    def train_step(self):
        self.step += 1
        if self.step % self.nb_interval == 0 and \
           len(self.buffer) >= self.batch_size:
            self.learn()

    def learn(self):
        batch = self.buffer.sample(self.batch_size)
        s, a, r, ns, d = zip(*batch)

        s = torch.tensor(np.stack(s), dtype=torch.float32).to(self.device)
        ns = torch.tensor(np.stack(ns), dtype=torch.float32).to(self.device)
        a = torch.tensor(a, dtype=torch.int64).to(self.device)
        r = torch.tensor(r, dtype=torch.float32).to(self.device)
        d = torch.tensor(d, dtype=torch.float32).to(self.device)

        with torch.no_grad():
            next_q = self.target_net(ns).max(1)[0]
            n_step_gamma = self.gamma ** self.n_step
            q_target = r + n_step_gamma * next_q * (1.0 - d)

        q_expected = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

        loss = self.loss_fn(q_expected, q_target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.update_target()

        debugger.log('train_loss', loss.item())
        debugger.log('epsilon', self.epsilon)

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def soft_update(self):
        for tparam, pparam in zip(self.target_net.parameters(), self.policy_net.parameters()):
            tparam.data.copy_(tparam.data * (1.0 - self.tau) + pparam.data * self.tau)

    def update_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

class EnhWorker:
    def __init__(self, cfg):
        self.cfg = cfg
        self.step = 0
        self.n_step = cfg.n_step
        self.state_dim = cfg.state_dim_ctrl
        self.action_dim = cfg.action_dim_ctrl

        self.gamma = cfg.gamma
        self.epsilon = cfg.epsilon_start
        self.epsilon_min = cfg.epsilon_min
        self.epsilon_decay = cfg.epsilon_decay
        self.batch_size = cfg.batch_size
        self.tau = cfg.tau
        self.device = resolve_torch_device()

        self.buffer = NStepReplayBuffer(cfg.buffer_capacity, self.n_step, self.gamma)

        self.policy_net = MultiHeadQNetwork(
            self.state_dim,
            self.action_dim,
            num_heads=4
        ).to(self.device)
        self.target_net = MultiHeadQNetwork(
            self.state_dim,
            self.action_dim,
            num_heads=4
        ).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg.learning_rate)

        self.scheduler = optim.lr_scheduler.ExponentialLR(
            self.optimizer, gamma=cfg.learning_rate_decay
        )

        self.loss_fn = nn.MSELoss()
        self.nb_interval = cfg.nb_interval

    def select_action(self, state):
        if random.random() < self.epsilon:
            return [random.randint(0, self.action_dim - 1) for _ in range(4)]

        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            qvals = self.policy_net(state)   # (1, 4, actions)
            actions = qvals.argmax(dim=2).squeeze(0)
            return actions.tolist()
        
    def remember(self, s, action, r, ns, done):
        self.buffer.push(s, action, r, ns, done)

    def train_step(self):
        self.step += 1
        if self.step % self.nb_interval == 0 and \
           len(self.buffer) >= self.batch_size:
            self.learn()

    def learn(self):
        batch = self.buffer.sample(self.batch_size)
        s, a, r, ns, d = zip(*batch)

        s = torch.tensor(np.stack(s), dtype=torch.float32).to(self.device)
        ns = torch.tensor(np.stack(ns), dtype=torch.float32).to(self.device)
        a = torch.tensor(np.stack(a), dtype=torch.long).to(self.device)
        r = torch.tensor(r, dtype=torch.float32).to(self.device).unsqueeze(1)
        d = torch.tensor(d, dtype=torch.float32).to(self.device).unsqueeze(1)

        # ---------- target ----------
        with torch.no_grad():
            next_q = self.target_net(ns)
            q_max_next = next_q.max(dim=2)[0]
            n_step_gamma = self.gamma ** self.n_step
            q_target = r + n_step_gamma * q_max_next * (1 - d)

        # ---------- expected ----------
        q_expected = self.policy_net(s).gather(2, a.unsqueeze(2)).squeeze(2)

        loss = self.loss_fn(q_expected, q_target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.scheduler.step()
        self.update_target()

        debugger.log('train_loss_enh', loss.item())
        debugger.log('epsilon_enh', self.epsilon)

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def soft_update(self):
        for tparam, pparam in zip(self.target_net.parameters(), self.policy_net.parameters()):
            tparam.data.copy_(
                tparam.data * (1.0 - self.tau) + pparam.data * self.tau
            )

    def update_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def reset_step(self):
        self.step = 0

In [37]:
class FeatureAdapter:
    def __init__(self, env: Any, cfg: Any):
        self.env = env
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short * cfg.viewport)
        self.tile_hist_long = deque(maxlen=cfg.h_long * cfg.viewport)
        self.tiles_hist_short = deque(maxlen=cfg.h_short)
        self.tiles_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)
        self.tiles_freq_short = defaultdict(int)
        self.tiles_freq_long = defaultdict(int)

        self.ch_video_hist = deque(maxlen=cfg.h_long)
        self.ch_viewport_hist = deque(maxlen=cfg.h_long)

        self.all_video_hist = []

    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tiles_hist_short,
            self.tiles_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
            self.ch_video_hist,
            self.ch_viewport_hist,            
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tiles_freq_short,
            self.tiles_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )

        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()

        self.all_video_hist = []

    def update_history(self, vid: int, tiles: list[int]):       
        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        tiles = tuple(tiles) if tiles is not None else None

        if tiles is None:
            return

        self._update_window(self.tiles_hist_short, self.tiles_freq_short, tiles)
        self._update_window(self.tiles_hist_long, self.tiles_freq_long, tiles)

        for tile in tiles:
            self._update_window(self.tile_hist_short, self.tile_freq_short, (vid, tile))
            self._update_window(self.tile_hist_long, self.tile_freq_long, (vid, tile))

    def update_history_single(self, item):
        if isinstance(item, int):
            self._update_window(self.video_hist_short, self.video_freq_short, item)
            self._update_window(self.video_hist_long, self.video_freq_long, item)
        elif isinstance(item, tuple) and len(item) == 2:
            self._update_window(self.tile_hist_short, self.tile_freq_short, item)
            self._update_window(self.tile_hist_long, self.tile_freq_long, item)
        elif isinstance(item, tuple) and all(isinstance(i, int) for i in item):
            self._update_window(self.tiles_hist_short, self.tiles_freq_short, item)
            self._update_window(self.tiles_hist_long, self.tiles_freq_long, item)

    def update_ch_history(self, vid: int, viewport: list[int]) -> None:
        self.all_video_hist.append(vid)

        video_cache_index = self.env.mec_cache.policy.video_idx
        tile_cache_index = self.env.mec_cache.policy.tile_idx

        hit = 1 if vid in video_cache_index else 0
        self.ch_video_hist.append(hit)

        if not hit:
            viewport_vector = [0,0,0,0]
        else:
            idx = video_cache_index.index(vid)
            cached_tiles = tile_cache_index[idx]
            viewport_vector = [1 if tile in cached_tiles else 0 for tile in viewport]

        self.ch_viewport_hist.append(viewport_vector)

    def compute_reward_layer_0(self, window_size: int = None) -> float:
        if window_size is None:
            window_size = len(self.ch_video_hist)

        ch_video_list = list(self.ch_video_hist)[-window_size:]
        psnr_layer_0 = 30 * sum(ch_video_list)
        
        return psnr_layer_0 / len(ch_video_list) if ch_video_list else 0

    def compute_reward_layer_1(self, window_size: int = None) -> float:
        if window_size is None:
            window_size = len(self.ch_viewport_hist)

        ch_viewport_list = list(self.ch_viewport_hist)[-window_size:]
        psnr_layer_1 = 2.5 * sum(sum(viewport) for viewport in ch_viewport_list)

        return psnr_layer_1 / len(ch_viewport_list) if ch_viewport_list else 0

    def compute_reward(self, window_size: int = None) -> float:
        if window_size is None:
            window_size = len(self.ch_video_hist)
        
        ch_video_list = list(self.ch_video_hist)[-window_size:]
        ch_viewport_list = list(self.ch_viewport_hist)[-window_size:]

        psnr_layer_0 = 30 * sum(ch_video_list)
        psnr_layer_1 = 2.5 * sum(sum(viewport) for viewport in ch_viewport_list)

        total_items = len(ch_video_list)
        return (psnr_layer_0 + psnr_layer_1) / total_items if total_items > 0 else 0

    def compute_meta_potential(self) -> float:
        video_cache_index = self.env.mec_cache.policy.video_idx
        potential = 0.0

        if self.video_hist_long.maxlen > 0:
            for vid in video_cache_index:
                if vid != -1:
                    prob = self.video_freq_long.get(vid, 0) / self.video_hist_long.maxlen
                    potential += prob

        return potential

    def compute_ctrl_potential(self) -> float:
        video_cache_index = self.env.mec_cache.policy.video_idx
        tile_cache_index = self.env.mec_cache.policy.tile_idx
        potential = 0.0
        
        if self.tile_hist_long.maxlen > 0:
            for vid_idx, vid in enumerate(video_cache_index):
                if vid == -1: continue
                
                for tile in tile_cache_index[vid_idx]:
                    if tile != -1:
                        prob = self.tile_freq_long.get((vid, tile), 0) / self.tile_hist_long.maxlen
                        potential += prob

        return potential

    def _update_window(self, hist_queue: deque, freq_dict: Dict, item):
        if len(hist_queue) == hist_queue.maxlen:
            old_item = hist_queue.popleft()
            freq_dict[old_item] -= 1
            if freq_dict[old_item] == 0:
                del freq_dict[old_item]
        hist_queue.append(item)
        freq_dict[item] += 1


In [38]:
class NetworkAdapter:
    def __init__(self, cfg: Any, env: Any, feature_adapter: Any):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.C = self.cfg.cache_size  # paper's cache capacity (videos)
        self.k = self.cfg.viewport    # paper's tiles per video (enhancement)

        print(f"NetworkAdapter initialized with capacity: {self.C} videos, {self.k} tiles per video")

    def build_observation(self, req) -> np.ndarray:
        
        if req is None:
            return np.zeros(self.C * 2 + 2, dtype=np.float32), np.zeros(self.k * 2 + self.k * 2, dtype=np.float32)
        
        vid = req["video"]
        viewport = req["viewport"]
        
        video_cache_index = self.env.mec_cache.policy.video_idx
        tile_cache_index = self.env.mec_cache.policy.tile_idx
        
        x_s = np.zeros(self.C, dtype=np.float32)
        x_l = np.zeros(self.C, dtype=np.float32)

        for vid_i, v in enumerate(video_cache_index):
            if v == -1:
                continue
            x_s[vid_i] = self.features.video_freq_short.get(v, 0) / self.features.video_hist_short.maxlen
            x_l[vid_i] = self.features.video_freq_long.get(v, 0) / self.features.video_hist_long.maxlen

        z_s = np.array(
            [self.features.video_freq_short.get(vid, 0) / self.features.video_hist_short.maxlen], dtype=np.float32
        )
        z_l = np.array(
            [self.features.video_freq_long.get(vid, 0) / self.features.video_hist_long.maxlen], dtype=np.float32
        )

        features_0 = np.concatenate([x_s, x_l, z_s, z_l], axis=0)

        vid_idx = self.env.mec_cache.get_video_cache_idx(vid)
        vp_tiles = tile_cache_index[vid_idx]
        
        y_s = np.zeros(self.k, dtype=np.float32)
        y_l = np.zeros(self.k, dtype=np.float32)

        for til_i, t in enumerate(vp_tiles):
            if t == -1:
                continue
            y_s[til_i] = self.features.tile_freq_short.get((vid, t), 0) / self.features.tile_hist_short.maxlen
            y_l[til_i] = self.features.tile_freq_long.get((vid, t), 0) / self.features.tile_hist_long.maxlen

        z_s = np.zeros(len(viewport), dtype=np.float32)
        z_l = np.zeros(len(viewport), dtype=np.float32)

        for i, tile in enumerate(viewport):
            z_s[i] = self.features.tile_freq_short.get((vid, tile), 0) / self.features.tile_hist_short.maxlen
            z_l[i] = self.features.tile_freq_long.get((vid, tile), 0) / self.features.tile_hist_long.maxlen

        features_1 = np.concatenate([y_s, y_l, z_s, z_l], axis=0)
        return (features_0, features_1)

    def reset(self):
        obs, info = self.env.reset()
        self.features.reset_history()

        return obs, info
    
    def env_is_done(self) -> bool:
        return self.env.users_env.all_users_done()

In [39]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'epsilon',
            'lr'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': round(float(total_reward), 2),
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'lr': f"{agent.scheduler.get_last_lr()[0]:.10f}" if agent else None,
            'epsilon': round(float(agent.epsilon), 4) if agent else None
        })

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

def build_latency_model(cfg):
    """Build and return the MultiDULatencyModel."""
    from importnb import Notebook
    with Notebook():
        from Labs.LatencyModel import MultiDULatencyModel
            
    P = cfg.n_nodes
    max_U = cfg.n_users

    return MultiDULatencyModel(
        P=P,
        max_U=max_U,
        R_M_D=80e6,
        R_C_M=125e6,
        mu=2e7,
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),
        rhoT_p=[0.2],
        lambda_p=[0.05],
        du_fixed_delay=0.001,
        mec_fixed_delay=0.005,
        cloud_fixed_delay=0.1
    )

def build_environment(cfg):
    """Construct the full multi-component environment wrapper."""
    from importnb import Notebook
    with Notebook():
        from Labs.CacheEngine import CacheEngineEnv
        from Labs.UserRequest import UserRequestEvents
        from Labs.EnvWrapper import EnvWrapper

    du_caches = []

    # DRL Caching Policy
    policy = DrlPolicy(cfg=cfg)

    # MEC Cache Engine
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity,
        policy=policy
    )

    # User request generator
    users_env = UserRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        arrival_rate=cfg.arrival_rate,
        zipf_alpha=cfg.zipf_alpha
    )

    # Latency Model
    latency_model = build_latency_model(cfg)

    # Wrapping all into the main training environment
    return EnvWrapper(
        cfg=cfg,
        n=cfg.n,
        m=cfg.m,
        n_layers=cfg.n_layers,
        users_env=users_env,
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=latency_model,
        theta=cfg.theta,
        lam=cfg.lam,
        max_steps=cfg.max_steps,
        prefetch_fn=lambda cache, action: cache.drl_prefetching_focus(action),
        reward_fn=lambda env, reqs: env.compute_reward(reqs),
        debugger=debugger
    )

In [40]:
class TrainingManager:
    def __init__(self, cfg):
        self.cfg = cfg
        self.env = build_environment(cfg)
        self.base_agent = BaseWorker(cfg)
        self.enh_agent = EnhWorker(cfg)

        self.feature_adapter = FeatureAdapter(self.env, cfg)
        self.net_adapter = NetworkAdapter(
            self.cfg, self.env, self.feature_adapter
        )

        self.ep_rewards = []
        self.ep_cache_hits = []
        self.ep_cache_misses = []

        self.k_base = 600.0  # Scales probability delta up to PSNR 30 bound
        self.k_enh = 250.0  # Scales probability delta up to PSNR 10 bound

    def select_action(self, state):
        state_base, state_enh = state
        action_base = self.base_agent.select_action(state_base)
        action_enh = self.enh_agent.select_action(state_enh)

        return np.concatenate([[action_base], action_enh], axis=0) 
    
    def train_fn(self):

        total_reward = 0.0
        cache_hits = cache_misses = 0
        base_hits = base_misses = 0
        enh_hits = enh_misses = 0

        _, info = self.net_adapter.reset()

        self.env.warmup_phase(self.net_adapter)

        for step in count():
            req = info["user_request"]
            state = self.net_adapter.build_observation(req)

            # --- Action Selection --- 
            action = self.select_action(state)
                        
            # --- Environment Step ---
            obs, reward, done, info = self.env.step(
                action, req, self.net_adapter
            )

            nxt_req = info["user_request"]

            reward_0 = info["reward_layer_0"]
            reward_1 = info["reward_layer_1"]
            prefetch_base = info["prefetch_base"]
            prefetch_enh = info["prefetch_enh"]
                    
            state_base, state_enh = state
            next_state_base, next_state_enh = self.net_adapter.build_observation(nxt_req)
    
            if prefetch_base:
                self.base_agent.remember(
                    state_base, action[0], reward_0, next_state_base, done
                )
                self.base_agent.train_step()

            if prefetch_enh:
                self.enh_agent.remember(
                    state_enh, action[1:], reward_1, next_state_enh, done
                )
                self.enh_agent.train_step()

            # --- Update Metrics ---
            delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
            total_reward += delta_r
            cache_hits += bs_hits + e_hits
            cache_misses += bs_miss + e_miss
            base_hits += bs_hits
            base_misses += bs_miss
            enh_hits += e_hits
            enh_misses += e_miss

            if done:
                break
            
        return total_reward, cache_hits, cache_misses, base_hits, base_misses, enh_hits, enh_misses

In [ ]:
def train_fn(trial_config):
    print("RL training started with config:", trial_config)
    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")

    run_cfg = config.Config()
    run_cfg.filename = cfg.filename
    for key, value in trial_config.items():
        if hasattr(run_cfg, key):
            setattr(run_cfg, key, value)

    debug_path = os.path.join(run_cfg.path_results, date_dir)
    os.makedirs(debug_path, exist_ok=True)
    
    manager = TrainingManager(run_cfg)
    logger = SummaryWriter(log_dir=debug_path)
    
    for ep in range(run_cfg.n_episodes):
        total_reward, hits, misses, bs_hits, bs_miss, enh_hits, enh_miss = manager.train_fn()

        manager.base_agent.update_epsilon()
        manager.enh_agent.update_epsilon()

        save_training_results(
            path_=run_cfg.path_results,
            filename=run_cfg.filename,
            ep=ep,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=manager.base_agent
        )
        
        debugger.log('lr', manager.base_agent.scheduler.get_last_lr()[0])
        debugger.log('epsilon', manager.base_agent.epsilon)
        debugger.log('lr_enh', manager.enh_agent.scheduler.get_last_lr()[0])
        debugger.log('epsilon_enh', manager.enh_agent.epsilon)
        
        hr = hits / (hits + misses + 1e-9)
        bhr = bs_hits / (bs_hits + bs_miss + 1e-9)
        ehr = enh_hits / (enh_hits + enh_miss + 1e-9)
        
        # print(
        #     f"--- Episode {ep} | R: {total_reward:.2f} | "
        #     f"HR: {hr:.2f} | "
        #     f"BHR: {bhr:.2f} | "
        #     f"EHR: {ehr:.2f} ---"
        # )

        logger.add_scalar("reward/total", total_reward, ep)
        logger.add_scalar("cache/hit_rate", hr, ep)
        logger.add_scalar("cache/base_hit_rate", bhr, ep)
        logger.add_scalar("cache/enh_hit_rate", ehr, ep)

        tune.report({
            "episode": int(ep),
            "total_reward": float(total_reward),
            "hit_rate": float(hr),
            "base_hit_rate": float(bhr),
            "enh_hit_rate": float(ehr),
        })

        debugger.save_results(filepath=f"{debug_path}/debug_ep{ep}")
        debugger.clear()

        print("-" * 50)

    logger.close()

In [ ]:
if __name__ == "__main__":
    if ray.is_initialized():
        ray.shutdown()

    ray.init(
        ignore_reinit_error=True,
        runtime_env={
            "working_dir": PROJECT_ROOT,
            "excludes": RAY_EXCLUDES,
            "env_vars": {
                "PYTHONPATH": PYTHONPATH_VALUE,
                "CUDA_VISIBLE_DEVICES": "",
            },
        },
    )

    scheduler = PopulationBasedTraining(
        metric="total_reward",
        mode="max",
        perturbation_interval=160,
        hyperparam_mutations={
            "lr": lambda: 10 ** np.random.uniform(-4, -1),
            "batch_size": [64, 128, 256],
            "optimizer": ["adam", "sgd"],
        },
    )
    
    base_config = {
        "lr": tune.loguniform(1e-4, 1e-1),
        "batch_size": tune.choice([64, 128, 256]),
        "optimizer": tune.choice(["adam", "sgd"]),
    }

    # Use short local paths on Windows to avoid TensorBoardX event file creation failures
    tune_storage_path = os.path.join(PROJECT_ROOT, "ray_runs")
    os.makedirs(tune_storage_path, exist_ok=True)

    def short_trial_dirname_creator(trial):
        return f"t_{trial.trial_id}"

    def short_trial_name_creator(trial):
        return f"trial_{trial.trial_id}"

    print("== Starting Ray Tune with Population Based Training scheduler ==")

    for seed in range(0, 4):
        run_config = dict(base_config)
        run_config["seed"] = seed

        analysis = tune.run(
            train_fn,
            scheduler=scheduler,
            num_samples=4,
            reuse_actors=True,
            config=run_config,
            name=f"mmsp_pbt_s{seed}",
            storage_path=tune_storage_path,
            trial_dirname_creator=short_trial_dirname_creator,
            trial_name_creator=short_trial_name_creator,
            verbose=1,
            progress_reporter=tune.CLIReporter(
                max_progress_rows=10,
                max_report_frequency=30,
                print_intermediate_tables=True,
                metric_columns=[
                    "episode", "total_reward", "hit_rate", "base_hit_rate", "enh_hit_rate"
                ]
            )
        )
        
        all_dfs = analysis.trial_dataframes
        names = list(all_dfs.keys())

        results = pd.DataFrame()
        for i in range(min(4, len(names))):
            df = all_dfs[names[i]].copy()
            df['sample_num'] = i 
            results = pd.concat([results, df]).reset_index(drop=True)

        dir = f"ray_tune_file_size4_method_env_default_max_160_batch"
        exist_dir = os.path.expanduser('~/data/' + dir)
        if not(os.path.exists(exist_dir)):
            os.makedirs(exist_dir)

        result_dir1 = os.path.expanduser('~/data/')
        result_dir2 = f"{result_dir1}{dir}/seed{seed}.csv"
        results.to_csv(result_dir2, index=False)

2026-03-02 07:29:05,921	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2026-03-02 07:29:05,921	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [WindowsPath('c:/Users/es25591/Workspace/CacheVideoPredict360/.gitignore')]
2026-03-02 07:29:06,030	INFO packaging.py:691 -- Creating a file package for local module 'c:\Users\es25591\Workspace\CacheVideoPredict360'.
2026-03-02 07:29:06,030	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [WindowsPath('c:/Users/es25591/Workspace/CacheVideoPredict360/.gitignore')]
2026-03-02 07:29:06,030	WARNING packaging.py:516 -- File c:\Users\es25591\Workspace\CacheVideoPredict360\Dataset.zip is very large (24.23MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['c:\Users\es25591\Workspace\CacheVideoPredict360\Dataset.zip']})`
2026-03-02 07:29:06,240	INFO packaging.py:463 -- Pushing file package 'gcs://_

== Starting Ray Tune with Population Based Training scheduler ==
== Status ==
Current time: 2026-03-02 07:29:08 (running for 00:00:00.14)
PopulationBasedTraining: 0 checkpoints, 0 perturbs
Logical resource usage: 0/28 CPUs, 0/0 GPUs
Result logdir: C:/Users/es25591/AppData/Local/Temp/ray/session_2026-03-02_07-29-00_573575_46724/artifacts/2026-03-02_07-29-07/mmsp_pbt_s0/driver_artifacts
Number of trials: 4/4 (4 PENDING)
+-------------------+----------+-------+--------------+-------------+-------------+
| Trial name        | status   | loc   |   batch_size |          lr | optimizer   |
|-------------------+----------+-------+--------------+-------------+-------------|
| trial_7fc64_00000 | PENDING  |       |          128 | 0.000532323 | sgd         |
| trial_7fc64_00001 | PENDING  |       |          128 | 0.00524395  | sgd         |
| trial_7fc64_00002 | PENDING  |       |          256 | 0.0416807   | sgd         |
| trial_7fc64_00003 | PENDING  |       |          256 | 0.000435463 | sgd 

(raylet) Stack (most recent call first):
(raylet)   File "c:\Users\es25591\Workspace\CacheVideoPredict360\venv\Lib\site-packages\ray\_private\worker.py", line 625 in job_logging_config
(raylet)   File "c:\Users\es25591\Workspace\CacheVideoPredict360\venv\Lib\site-packages\ray\_private\worker.py", line 2800 in disconnect
(raylet)   File "c:\Users\es25591\Workspace\CacheVideoPredict360\venv\Lib\site-packages\ray\_private\worker.py", line 2132 in shutdown
(raylet)   File "c:\Users\es25591\Workspace\CacheVideoPredict360\venv\Lib\site-packages\ray\_private\worker.py", line 1144 in wrapper


(train_fn pid=25648) RL training started with config: {'lr': 0.00043546265820505037, 'batch_size': 256, 'optimizer': 'sgd', 'seed': 0}
(train_fn pid=38276) NetworkAdapter initialized with capacity: 50 videos, 4 tiles per video
(train_fn pid=17668) --------------------------------------------------
(train_fn pid=28916) RL training started with config: {'lr': 0.041680713705057874, 'batch_size': 256, 'optimizer': 'sgd', 'seed': 0} [repeated 3x across cluster]
(train_fn pid=28916) NetworkAdapter initialized with capacity: 50 videos, 4 tiles per video [repeated 3x across cluster]
(train_fn pid=38276) -------------------------------------------------- [repeated 12x across cluster]
(train_fn pid=17668) -------------------------------------------------- [repeated 12x across cluster]
(train_fn pid=17668) -------------------------------------------------- [repeated 12x across cluster]


2026-03-02 07:29:34,525	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to 'c:/Users/es25591/Workspace/CacheVideoPredict360/ray_runs/mmsp_pbt_s0' in 0.0150s.
2026-03-02 07:29:34,528	INFO tune.py:1041 -- Total run time: 26.60 seconds (26.56 seconds for the tuning loop).
2026-03-02 07:29:34,537	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


== Status ==
Current time: 2026-03-02 07:29:34 (running for 00:00:26.58)
PopulationBasedTraining: 0 checkpoints, 0 perturbs
Logical resource usage: 1.0/28 CPUs, 0/0 GPUs
Result logdir: C:/Users/es25591/AppData/Local/Temp/ray/session_2026-03-02_07-29-00_573575_46724/artifacts/2026-03-02_07-29-07/mmsp_pbt_s0/driver_artifacts
Number of trials: 4/4 (4 TERMINATED)
+-------------------+------------+-----------------+--------------+-------------+-------------+----------------+-----------+------------+-----------------+----------------+
| Trial name        | status     | loc             |   batch_size |          lr | optimizer   |   total_reward |   episode |   hit_rate |   base_hit_rate |   enh_hit_rate |
|-------------------+------------+-----------------+--------------+-------------+-------------+----------------+-----------+------------+-----------------+----------------|
| trial_7fc64_00000 | TERMINATED | 127.0.0.1:38276 |          128 | 0.000532323 | sgd         |        6221.13 |       

2026-03-02 07:30:00,200	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to 'c:/Users/es25591/Workspace/CacheVideoPredict360/ray_runs/mmsp_pbt_s1' in 0.0173s.
2026-03-02 07:30:00,200	INFO tune.py:1041 -- Total run time: 25.66 seconds (25.63 seconds for the tuning loop).
2026-03-02 07:30:00,216	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


== Status ==
Current time: 2026-03-02 07:30:00 (running for 00:00:25.65)
PopulationBasedTraining: 0 checkpoints, 0 perturbs
Logical resource usage: 1.0/28 CPUs, 0/0 GPUs
Result logdir: C:/Users/es25591/AppData/Local/Temp/ray/session_2026-03-02_07-29-00_573575_46724/artifacts/2026-03-02_07-29-34/mmsp_pbt_s1/driver_artifacts
Number of trials: 4/4 (4 TERMINATED)
+-------------------+------------+-----------------+--------------+-------------+-------------+----------------+-----------+------------+-----------------+----------------+
| Trial name        | status     | loc             |   batch_size |          lr | optimizer   |   total_reward |   episode |   hit_rate |   base_hit_rate |   enh_hit_rate |
|-------------------+------------+-----------------+--------------+-------------+-------------+----------------+-----------+------------+-----------------+----------------|
| trial_8fa2c_00000 | TERMINATED | 127.0.0.1:11488 |          128 | 0.000639499 | sgd         |        7680.18 |       